# Visual Wake Word - FP4 Model Evaluation

This notebook evaluates the original and FP4-quantized VWW models on COCO minival.

**Before running:**
1. Upload `val2014.zip` and `annotations_trainval2014.zip` to your Google Drive
2. Right-click each file → Share → Copy link
3. Paste the links in the cell below

In [ ]:
#@title 1. Configure Google Drive Links { display-mode: "form" }

#@markdown Paste your Google Drive sharing links below:
val2014_link = "https://drive.google.com/file/d/XXXXX/view?usp=sharing" #@param {type:"string"}
annotations_link = "https://drive.google.com/file/d/XXXXX/view?usp=sharing" #@param {type:"string"}

import re

def extract_gdrive_id(link):
    """Extract file ID from various Google Drive URL formats."""
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'^([a-zA-Z0-9_-]{20,})$',
    ]
    for p in patterns:
        m = re.search(p, link.strip())
        if m:
            return m.group(1)
    raise ValueError(f"Could not extract file ID from: {link}")

val2014_id = extract_gdrive_id(val2014_link)
annotations_id = extract_gdrive_id(annotations_link)
print(f"val2014 file ID: {val2014_id}")
print(f"annotations file ID: {annotations_id}")

In [ ]:
#@title 2. Install dependencies & clone repo
!pip install -q pycocotools opencv-python-headless gdown
!git clone https://github.com/amitmate/visualwakeword.git /content/vww 2>/dev/null || (cd /content/vww && git pull)
print("Done.")

In [ ]:
#@title 3. Download data from Google Drive
import os
import gdown

os.makedirs('/content/coco/raw-data/annotations', exist_ok=True)
os.makedirs('/content/coco/raw-data/val2014', exist_ok=True)

# Download annotations
if not os.path.exists('/content/coco/annotations_trainval2014.zip'):
    print("Downloading annotations...")
    gdown.download(id=annotations_id, output='/content/coco/annotations_trainval2014.zip', quiet=False)
else:
    print("Annotations zip already exists, skipping download.")

# Download val2014 images
if not os.path.exists('/content/coco/val2014.zip'):
    print("Downloading val2014 images (this may take a while)...")
    gdown.download(id=val2014_id, output='/content/coco/val2014.zip', quiet=False)
else:
    print("val2014 zip already exists, skipping download.")

print("Downloads complete!")
!ls -lh /content/coco/*.zip

In [ ]:
#@title 4. Extract data
import zipfile

# Extract annotations
if not os.path.exists('/content/coco/raw-data/annotations/instances_val2014.json'):
    print("Extracting annotations...")
    with zipfile.ZipFile('/content/coco/annotations_trainval2014.zip', 'r') as z:
        # Only extract instances_val2014.json
        for f in z.namelist():
            if 'instances_val2014.json' in f:
                z.extract(f, '/content/coco/raw-data')
                print(f"  Extracted: {f}")
else:
    print("Annotations already extracted.")

# Extract val2014 images
val2014_dir = '/content/coco/raw-data/val2014'
existing = len(os.listdir(val2014_dir))
if existing < 100:
    print("Extracting val2014 images (this takes a few minutes)...")
    !unzip -q -o /content/coco/val2014.zip -d /content/coco/raw-data/
    print(f"  Extracted {len(os.listdir(val2014_dir))} images.")
else:
    print(f"val2014 already extracted ({existing} images).")

# Verify
print("\nVerification:")
!ls /content/coco/raw-data/annotations/instances_val2014.json
!ls /content/coco/raw-data/val2014/ | head -5
!echo "Total images: $(ls /content/coco/raw-data/val2014/ | wc -l)"

In [ ]:
#@title 5. Copy minival IDs
import shutil
shutil.copy('/content/vww/mscocominival.txt', '/content/coco/mscocominival.txt')
!wc -l /content/coco/mscocominival.txt

In [ ]:
#@title 6. Run Evaluation
%cd /content/vww
!python eval_fp4_coco.py \
    --data-dir /content/coco/raw-data \
    --minival-ids /content/coco/mscocominival.txt \
    --h5-model modelVisualWakeWord.h5 \
    --tflite-model modelVisualWakeWord.tflite \
    --fp4-bin compressed_models/model_mxfp4.fp4bin \
    --fp4-tflite compressed_models/model_mxfp4_int8.tflite

## Alternative: Mount Google Drive directly

If you prefer to mount your Drive instead of using sharing links, run the cell below and update the paths accordingly.

In [ ]:
#@title (Optional) Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Then update paths, e.g.:
# # !cp /content/drive/MyDrive/val2014.zip /content/coco/
# # !cp /content/drive/MyDrive/annotations_trainval2014.zip /content/coco/